# MiniVLMDocEval — Colab runner (terminal)

This is the **only** notebook. It is a thin terminal: pull the latest code from GitHub and run a `.py` script on the GPU. All real logic lives in `.py` files, edited locally (by Claude) and synced via git.

**Loop:** Claude edits scripts locally → push → you re-run the *Run* cell (`git pull && python ...`).

One-time per session: run the **GPU+clone**, **Drive mount**, and **setup** cells. All eval outputs (predictions + summary tables) are written to Google Drive (`WORK_DIR`) so they persist across sessions and `--reuse` can resume.

Set runtime to **T4 GPU** before running.

In [ ]:
# --- One-time per session: GPU + clone ---
import os
REPO_URL = 'https://github.com/SajjadPSavoji/MiniVLMDocEval.git'
REPO = '/content/MiniVLMDocEval'

!nvidia-smi -L || echo 'NO GPU — Runtime > Change runtime type > T4 GPU'
if not os.path.isdir(REPO):
    assert REPO_URL, 'Set REPO_URL'
    !git clone $REPO_URL $REPO
%cd $REPO

In [ ]:
# --- One-time per session: mount Google Drive for persistent outputs ---
from google.colab import drive
drive.mount('/content/drive')

import os
OUT_DIR = '/content/drive/MyDrive/MiniVLMDocEval/outputs'  # contains predictions/, summary/, logs/
os.makedirs(OUT_DIR, exist_ok=True)
print('output base ->', OUT_DIR)

In [ ]:
# --- One-time per session: install the engine (VLMEvalKit @ pinned commit) ---
# A few minutes. If imports break afterwards, RESTART RUNTIME and re-run the Run cell.
!bash setup.sh

In [ ]:
# --- Run cell: full eval -> Drive (predictions/, summary/, logs/). Re-run to iterate.
#     --reuse resumes automatically if a session dropped mid-run.
#     First time, validate small by adding `--data OCRBench` (3 models x 1k).
#     Drop it for the full 5-benchmark suite.
!git pull --quiet && python scripts/run_eval.py --out $OUT_DIR